# Assignment 1A — Part B · QLoRA Instruction Fine-Tuning
### Team 1AE — Variant 5 (Enterprise IT / ITSM) · Domain: **Cisco Product Documentation**

Extends the **CPT checkpoint from Part A** into an instruction-following
assistant for Cisco networking/config queries.

**Tasks**
- **B1** Instruction dataset creation — >=100 pairs, JSONL, 80/20 split (2)
- **B2** QLoRA fine-tuning — 3 adapter configs (r = 8 / 16 / 32) (2)
- **B3** Evaluation & comparative analysis (1)

> Requires a GPU (4-bit `bitsandbytes`). Run on **T4 (Colab)** or **A100 (BITS lab)**.


## 0 · Configuration

In [ ]:
# Point CPT_CKPT at the checkpoint saved by Part A (cpt_model/).
# If running Part B standalone, set it to the base model id instead.
CPT_CKPT   = "cpt_model"                        # from Part A Step 4
BASE_FALLBACK = "HuggingFaceTB/SmolLM2-360M"    # used if cpt_model/ absent
DATA_REPO  = "https://github.com/ajoish_cisco/LLM_ASSIGNMENT.git"
PDF_DIR    = "LLM_ASSIGNMENT/docs"
CORPUS_DIR = "domain_corpus"
JSONL      = "instruction_dataset.jsonl"
ADAPTER_DIR = "adapters"

MAX_STEPS  = 300      # per adapter (raise on A100)
LR         = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM = 4
MAX_SEQ_LEN = 1024

import os, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_SRC = CPT_CKPT if os.path.isdir(CPT_CKPT) else BASE_FALLBACK
print("Training from:", MODEL_SRC, "| device:", DEVICE)


In [ ]:
%pip install -q "transformers>=4.44" "datasets>=2.20" "peft>=0.12" "trl>=0.9" "bitsandbytes>=0.43" accelerate pypdf


## B1 · Instruction Dataset Creation  [2 marks]
Turn the cleaned Cisco corpus into >=100 `instruction`/`response` pairs by
pairing section headings (`Configuring ...`, `Restrictions for ...`,
`Information About ...`) with the explanatory text that follows, then split
80/20. (If `instruction_dataset.jsonl` already exists from the repo/Part A run,
this cell simply reloads it.)

In [ ]:
import os, re, json, random
from pathlib import Path

# Ensure corpus is available (clone + extract if needed)
if not os.path.isdir(CORPUS_DIR) or not any(Path(CORPUS_DIR).glob("*.txt")):
    if not os.path.isdir("LLM_ASSIGNMENT"):
        !git clone --depth 1 {DATA_REPO}
    %pip install -q pypdf
    from pypdf import PdfReader
    Path(CORPUS_DIR).mkdir(exist_ok=True)
    for pdf in sorted(Path(PDF_DIR).glob("*.pdf")):
        pages = []
        for pg in PdfReader(str(pdf)).pages:
            try: pages.append(pg.extract_text() or "")
            except Exception: pages.append("")
        (Path(CORPUS_DIR)/f"{pdf.stem}.txt").write_text("\n".join(pages), encoding="utf-8")


In [ ]:
HEADING_RE = re.compile(
    r"^(Configuring|Restrictions for|Information About|How to Configure|"
    r"Prerequisites for|Overview of|About|Understanding|Guidelines for|"
    r"Verifying|Troubleshooting)\b.*", re.IGNORECASE)

def heading_to_question(h):
    h = h.strip().rstrip(":").strip(); low = h.lower()
    if low.startswith("configuring"):
        return f"How do I configure {h[len('Configuring'):].strip()} on a Cisco device?"
    if low.startswith("how to configure"):
        return f"How do I configure {h[len('How to Configure'):].strip()} on a Cisco device?"
    if low.startswith("restrictions for"):
        return f"What are the restrictions for {h[len('Restrictions for'):].strip()}?"
    if low.startswith("prerequisites for"):
        return f"What are the prerequisites for {h[len('Prerequisites for'):].strip()}?"
    if low.startswith(("information about","about")):
        topic = re.sub(r"^(information about|about)\s*", "", h, flags=re.I).strip()
        return f"What is {topic} and how does it work?"
    if low.startswith("understanding"):
        return f"Explain {h[len('Understanding'):].strip()} in Cisco networking."
    if low.startswith("verifying"):
        return f"How do I {h[0].lower()+h[1:]}?"
    if low.startswith("troubleshooting"):
        return f"How do I troubleshoot {h[len('Troubleshooting'):].strip()}?"
    if low.startswith("guidelines for"):
        return f"What are the configuration guidelines for {h[len('Guidelines for'):].strip()}?"
    return f"Explain the following Cisco topic: {h}"

def looks_like_toc(p):
    low = p.lower()
    return ("on page" in low or "onpage" in low or
            ("configuration guide" in low and "cisco ios" in low) or
            bool(re.search(r"catalyst\s*9\d{3}\s*switches", low)) or
            p.count("\u2022") >= 2 or p.count("...") >= 2)

def space_ratio(p): return p.count(" ")/len(p) if p else 0.0


In [ ]:
pairs, random_seed = [], random.seed(0)
for f in sorted(Path(CORPUS_DIR).glob("*.txt")):
    lines = f.read_text(encoding="utf-8").split("\n"); i = 0
    while i < len(lines):
        line = lines[i].strip()
        if HEADING_RE.match(line) and 8 <= len(line) <= 90:
            body, j = [], i+1
            while j < len(lines) and len(" ".join(body)) < 900:
                nxt = lines[j].strip()
                if HEADING_RE.match(nxt): break
                if nxt: body.append(nxt)
                elif body: break
                j += 1
            resp = re.sub(r"\s+"," "," ".join(body)).strip()
            if len(resp) >= 120 and not looks_like_toc(resp) and space_ratio(resp) >= 0.10:
                pairs.append({"instruction": heading_to_question(line),
                              "response": resp[:900], "source": f.stem})
            i = j
        else:
            i += 1

seen, uniq = set(), []
for p in pairs:
    k = p["instruction"].lower()
    if k not in seen: seen.add(k); uniq.append(p)
random.shuffle(uniq); uniq = uniq[:600]
assert len(uniq) >= 100, f"Only {len(uniq)} pairs (<100)"

n_train = int(len(uniq)*0.8)
train, eval_ = uniq[:n_train], uniq[n_train:]
for s, rows in (("train",train),("eval",eval_)):
    for r in rows: r["split"] = s
with open(JSONL,"w",encoding="utf-8") as fh:
    for r in train+eval_: fh.write(json.dumps(r, ensure_ascii=False)+"\n")
print(f"Pairs: {len(uniq)} | train {len(train)} / eval {len(eval_)}")
print(json.dumps(train[0], indent=2)[:400])


## B2 · QLoRA Fine-Tuning — 3 Adapter Configurations  [2 marks]
Load the CPT model in 4-bit (NF4) and train **three** LoRA adapters with
`SFTTrainer` using the model's chat template.

| Adapter | r | alpha | target modules | expected effect |
|---|---|---|---|---|
| A (low)      | 8  | 16 | q_proj, v_proj          | faster; may underfit |
| B (balanced) | 16 | 32 | q_proj, v_proj          | good quality/cost |
| C (high)     | 32 | 32 | q_proj, v_proj, o_proj  | best quality; more VRAM |


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

tok = AutoTokenizer.from_pretrained(MODEL_SRC)
if tok.pad_token is None: tok.pad_token = tok.eos_token

# Fallback chat template for base models that lack one
if tok.chat_template is None:
    tok.chat_template = (
        "{% for m in messages %}{% if m['role']=='user' %}"
        "<|user|>\n{{ m['content'] }}\n<|assistant|>\n"
        "{% else %}{{ m['content'] }}{{ eos_token }}{% endif %}{% endfor %}")

ds = load_dataset("json", data_files=JSONL)["train"]
train_ds = ds.filter(lambda r: r["split"] == "train")
eval_ds  = ds.filter(lambda r: r["split"] == "eval")

def format_row(r):
    msgs = [{"role":"user","content":r["instruction"]},
            {"role":"assistant","content":r["response"]}]
    return tok.apply_chat_template(msgs, tokenize=False)
print(format_row(train_ds[0])[:300])


In [ ]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

ADAPTERS = {
    "A": dict(r=8,  alpha=16, targets=["q_proj","v_proj"]),
    "B": dict(r=16, alpha=32, targets=["q_proj","v_proj"]),
    "C": dict(r=32, alpha=32, targets=["q_proj","v_proj","o_proj"]),
}

# Pre-render each example into a single `text` field using the chat template.
import inspect
train_txt = train_ds.map(lambda r: {"text": format_row(r)})

def make_sft_config(**kw):
    """Build SFTConfig, tolerating the trl max_seq_length -> max_length rename."""
    valid = set(inspect.signature(SFTConfig.__init__).parameters)
    if "max_seq_length" in kw and "max_seq_length" not in valid:
        kw["max_length"] = kw.pop("max_seq_length")
    return SFTConfig(**{k: v for k, v in kw.items() if k in valid or k == "output_dir"})

def train_adapter(name, spec):
    print(f"\n=== Adapter {name}: r={spec['r']} alpha={spec['alpha']} "
          f"targets={spec['targets']} ===")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_SRC, quantization_config=bnb, torch_dtype=torch.bfloat16,
        device_map="auto")
    peft_cfg = LoraConfig(r=spec["r"], lora_alpha=spec["alpha"], lora_dropout=0.05,
        bias="none", task_type="CAUSAL_LM", target_modules=spec["targets"])
    out = f"{ADAPTER_DIR}/adapter_{name}"
    cfg = make_sft_config(output_dir=out, max_steps=MAX_STEPS,
        per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR, warmup_steps=10, logging_steps=20, save_strategy="no",
        bf16=True, report_to="none", max_seq_length=MAX_SEQ_LEN,
        dataset_text_field="text", gradient_checkpointing=True)
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_txt,
        peft_config=peft_cfg)
    trainer.train()
    trainer.model.save_pretrained(out); tok.save_pretrained(out)
    print("saved ->", out); return out

adapter_paths = {name: train_adapter(name, spec) for name, spec in ADAPTERS.items()}


## B3 · Evaluation & Comparative Analysis  [1 mark]
Run all three adapters on the same 3 Cisco-domain prompts and compare which
gives the most accurate, domain-relevant answer.

In [ ]:
from peft import PeftModel
EVAL_PROMPTS = [
    "How do I configure a VLAN on a Catalyst 9300 switch?",
    "What are the restrictions for configuring BGP on Nexus 9000?",
    "How does Cisco ISE posture assessment work?",
]

@torch.no_grad()
def answer(adapter_path, prompt, max_new_tokens=120):
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_SRC, quantization_config=bnb, torch_dtype=torch.bfloat16,
        device_map="auto")
    m = PeftModel.from_pretrained(base, adapter_path)
    msgs = [{"role":"user","content":prompt}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt").to(m.device)
    out = m.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                     pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    del base, m; torch.cuda.empty_cache(); return text


In [ ]:
import pandas as pd
rows = []
for p in EVAL_PROMPTS:
    row = {"prompt": p}
    for name, path in adapter_paths.items():
        row[f"adapter_{name}"] = answer(path, p)
    rows.append(row)
comparison = pd.DataFrame(rows)
comparison.to_csv("data/adapter_comparison.csv", index=False)
comparison


### Which adapter wins?
Inspect the table above and record, per prompt, which adapter produced the most
accurate and domain-relevant response.

- **Adapter A (r=8)** — fastest, lowest VRAM; may underfit longer procedural answers.
- **Adapter B (r=16)** — usually the best quality/cost trade-off.
- **Adapter C (r=32, +o_proj)** — highest capacity; best on complex multi-step
  Cisco configuration answers, at higher VRAM and training cost.

### Deliverables produced
- `instruction_dataset.jsonl` — >=100 pairs, 80/20 split
- `adapters/adapter_{A,B,C}/` — three trained QLoRA adapters
- `data/adapter_comparison.csv` — B3 comparison table
